# 02 – Data cleaning

**Goal:** turn the raw tables into simple, analysis-ready CSV files in `data/processed/`.

Problems found in notebook 01 and fixed here:
- Each source writes province names differently. Every table gets the official **SR-3 (NUTS-3) code** (e.g. `TR100` = Istanbul) and joins on that code.
- TÜİK values are stored as text, some with a decimal comma (`2,68`). They are converted to numbers.
- TÜİK tables repeat each value as a count (`Sayı`) and a share (`Yüzde`). Only counts are kept.
- The first registrations table mixes single months with year-to-date totals. Only single months are kept.
- About 10 EPDK station addresses do not end with a province name. They are fixed by hand where possible.
- The EPDK monthly report is a PDF. Its national figures are typed in, with the page number as source.

Output: one row per province (`province_analysis_table.csv`), one row per station (`charging_stations.csv`) and one row per month for Türkiye (`national_monthly.csv`).

In [1]:
import json
import os

import pandas as pd

RAW = "../data/raw/"
TUIK = RAW + "tuik/"
PROCESSED = "../data/processed/"

LATEST_MONTH = "2026-09"   # latest month in the TÜİK vehicle tables
YEAR_AGO = "2025-09"

Helpers from notebook 01, plus `to_number` for values with a decimal comma.

In [2]:
def read_tuik_csv(file_name):
    df = pd.read_csv(TUIK + file_name, sep=";", encoding="utf-8-sig", low_memory=False)
    df.columns = [col.split(" (")[0] for col in df.columns]
    return df


def to_number(values):
    # "2,68" -> 2.68
    return pd.to_numeric(values.astype(str).str.replace(",", ".", regex=False))


def standard_name(name):
    # Turkish-aware upper case: i -> İ and ı -> I must be handled before .upper()
    name = str(name).strip().replace("i", "İ").replace("ı", "I")
    return name.upper()


def read_jsonstat(file_name):
    with open(TUIK + file_name, encoding="utf-8") as f:
        js = json.load(f)
    dims = js["id"]
    labels = []
    for d in dims:
        cat = js["dimension"][d]["category"]
        index = cat["index"]
        order = index if isinstance(index, list) else sorted(index, key=index.get)
        labels.append([(code, cat["label"][code]) for code in order])
    index = pd.MultiIndex.from_product(labels, names=dims)
    values = pd.Series(js["value"]).astype(float)
    values.index = values.index.astype(int)
    df = pd.DataFrame({"value": values}).reindex(range(len(index)))
    df.index = index
    return df.dropna().reset_index()

## 1. Province list with official codes

The GDP JSON-stat file lists every statistical region with its code. Codes with five characters are the 81 provinces (SR-3), and the first three characters give the SR-1 region (12 regions), e.g. `TR100` Istanbul → `TR1` İstanbul region, `TR211` Tekirdağ → `TR2` Batı Marmara.

In [3]:
gdp = read_jsonstat("tuik_DF_UH_BH_KISI_BASI.json")
# Each dimension value is a (code, label) pair
area_labels = dict(gdp["REF_AREA"].unique())

provinces = pd.DataFrame(
    [(code, label) for code, label in area_labels.items() if len(code) == 5],
    columns=["province_code", "province"],
)
provinces["region_code"] = provinces["province_code"].str[:3]
provinces["region"] = provinces["region_code"].map(area_labels)

print("Provinces:", len(provinces), "| regions:", provinces["region_code"].nunique())
provinces.head()

Provinces: 81 | regions: 12


,province_code,province,region_code,region
0,TR100,İstanbul,TR1,İstanbul
1,TR212,Edirne,TR2,Batı Marmara
2,TR213,Kırklareli,TR2,Batı Marmara
3,TR211,Tekirdağ,TR2,Batı Marmara
4,TR221,Balıkesir,TR2,Batı Marmara


A small function adds the province code to any table, using the standardised name from notebook 01. Names that do not match stay empty, so they can be checked.

In [4]:
CODE_BY_NAME = dict(zip(provinces["province"].map(standard_name), provinces["province_code"]))


def add_province_code(df, name_col):
    df = df.copy()
    df["province_code"] = df[name_col].map(standard_name).map(CODE_BY_NAME)
    return df

## 2. Registered vehicles by province (ILLER_V3)

Three columns per province:
- `cars`: registered cars in September 2026,
- `all_vehicles`: all registered motor vehicles in September 2026,
- `cars_2025_09`: registered cars one year earlier, to measure growth of the car fleet.

The Türkiye row is kept apart for the checks in section 9.

In [5]:
prov = read_tuik_csv("tuik_DF_MOTORLU_KARA_TASIT_ILLER_V3.csv")
prov = prov[prov["Ölçü Birimi"] == "Sayı"].copy()
prov["value"] = to_number(prov["Gözlem Değeri"])
prov = add_province_code(prov, "Coğrafi Kapsam")


def vehicles_in(month, vehicle_type, name):
    rows = prov[(prov["Zaman"] == month) & (prov["Araç Türü"] == vehicle_type) & prov["province_code"].notna()]
    return rows.set_index("province_code")["value"].rename(name)


vehicles = pd.concat([
    vehicles_in(LATEST_MONTH, "Otomobil", "cars"),
    vehicles_in(LATEST_MONTH, "Toplam", "all_vehicles"),
    vehicles_in(YEAR_AGO, "Otomobil", "cars_2025_09"),
], axis=1).reset_index()

turkiye_cars = prov.loc[(prov["Coğrafi Kapsam"] == "Türkiye") & (prov["Zaman"] == LATEST_MONTH)
                        & (prov["Araç Türü"] == "Otomobil"), "value"].iloc[0]
print(vehicles.shape)
vehicles.head()

(81, 4)


,province_code,cars,all_vehicles,cars_2025_09
0,TR100,4210192,6499506,3986817
1,TR212,89972,217307,85394
2,TR213,79044,186039,75278
3,TR211,199625,402798,188133
4,TR221,289797,723947,272336


## 3. New cars registered by province (KAYDI_YAPILAN_IL_V2)

Rows whose month label contains a dash (e.g. `Ocak-Eylül`) are year-to-date totals and are dropped. From the single months:
- `new_cars_2025`: cars registered for the first time in 2025,
- `new_cars_2026_jan_sep`: the same for January–September 2026.

The monthly Türkiye total is kept for the national table (section 8).

In [6]:
reg = read_tuik_csv("tuik_DF_MOTORLU_KARA_TASIT_KAYDI_YAPILAN_IL_V2.csv")
reg = reg[(reg["Ölçü Birimi"] == "Sayı") & (reg["Araç Türü"] == "Otomobil")
          & ~reg["Ay"].str.contains("-")].copy()
reg["value"] = to_number(reg["Gözlem Değeri"])
reg = add_province_code(reg, "Coğrafi Kapsam")

reg_prov = reg[reg["province_code"].notna()]
new_cars = pd.concat([
    reg_prov[reg_prov["Zaman"].str.startswith("2025")].groupby("province_code")["value"].sum().rename("new_cars_2025"),
    reg_prov[reg_prov["Zaman"].between("2026-01", LATEST_MONTH)].groupby("province_code")["value"].sum()
        .rename("new_cars_2026_jan_sep"),
], axis=1).reset_index()

new_cars_tr = (reg[reg["Coğrafi Kapsam"] == "Türkiye"][["Zaman", "value"]]
               .rename(columns={"Zaman": "month", "value": "new_cars"}))

print("Months since Jan 2025 per province:", reg_prov[reg_prov["Zaman"] >= "2025-01"].groupby("province_code").size().unique())
print("Türkiye, Jan-Sep 2026:", new_cars_tr.loc[new_cars_tr["month"].between("2026-01", LATEST_MONTH), "new_cars"].sum())
new_cars.head()

Months since Jan 2025 per province: [21]
Türkiye, Jan-Sep 2026: 665220


,province_code,new_cars_2025,new_cars_2026_jan_sep
0,TR100,429236,264974
1,TR211,12611,7349
2,TR212,4738,2456
3,TR213,4042,2192
4,TR221,13489,7637


Every province has all 21 months since January 2025, and January–September 2026 gives 665,220 new cars, the figure in the TÜİK press release. (A few small provinces such as Ardahan and Hakkari miss one or two months in 2005–2009, probably months with no registrations. These years are not used.)

## 4. EPDK charging stations

The province is the last part of the address, after the final `/`. Eleven addresses (all Otojet stations) do not end with a province name. Six of them can be placed with confidence from the station name or a known address:

| Station | Clue | Province |
| --- | --- | --- |
| ŞRJ/3660, ŞRJ/3686 | "Bucak Merkez", Karacaahmet Mahallesi | Burdur |
| ŞRJ/3872 | name "Buhari Çorlu", Önerler Mahallesi | Tekirdağ |
| ŞRJ/4044 | name "Altuğ Çiğli", Atatürk OSB | İzmir |
| ŞRJ/5647 | name "Alp Akaryakıt Konya", Konya Ereğli Caddesi | Konya |
| ŞRJ/5706 | Varsaklar Mahallesi, Adana Caddesi (Kozan; same street in OPET station list) | Adana |

The other five (ŞRJ/3374, 3635, 3637, 3652, 5383) have no clear province. They are kept in the station file with an empty province and left out of province counts (5 of 12,760 public stations, 0.04%).

Twenty stations have **no address at all**. One is a university building (`İST.ÜNİV.ASTRONOMİ FAKÜLTESİ`, placed in İstanbul). The other 19 are **mobile charging vans** (Skoda and EVbee, e.g. "Skoda Mobil Şarj - Ankara"). They move between places, so they get `mobile = True`, no province, and are left out of province counts.

In [7]:
stations = pd.read_csv(RAW + "epdk_sarj_istasyonlari_2026-10-30.csv", encoding="utf-8-sig")
stations["province_name"] = stations["adres"].str.split("/").str[-1].str.strip()

ADDRESS_FIX = {
    "ŞRJ/3660": "Burdur", "ŞRJ/3686": "Burdur", "ŞRJ/3872": "Tekirdağ",
    "ŞRJ/4044": "İzmir", "ŞRJ/5647": "Konya", "ŞRJ/5706": "Adana", "ŞRJ/21392": "İstanbul",
}
fix = stations["istasyon_no"].isin(ADDRESS_FIX)
stations["mobile"] = stations["adres"].isna() & ~fix
stations.loc[fix, "province_name"] = stations.loc[fix, "istasyon_no"].map(ADDRESS_FIX)
stations = add_province_code(stations, "province_name")

stations = stations.rename(columns={
    "istasyon_no": "station_id",
    "istasyon_adi": "station_name",
    "marka": "brand",
    "sarj_agi_isletmecisi": "network_operator",
    "sarj_istasyonu_isletmecisi": "station_operator",
    "adres": "address",
})
stations["access"] = stations["hizmet_sekli"].map({"Halka Açık": "public", "Özel": "private"})
stations = stations[["station_id", "station_name", "access", "mobile", "brand", "network_operator",
                     "station_operator", "address", "province_code"]]

print("Mobile chargers:", stations["mobile"].sum())
print("Other stations without a province:")
print(stations.loc[stations["province_code"].isna() & ~stations["mobile"], ["station_id", "access"]].to_string(index=False))

Mobile chargers: 19
Other stations without a province:
station_id access
  ŞRJ/3374 public
  ŞRJ/3635 public
  ŞRJ/3637 public
  ŞRJ/3652 public
  ŞRJ/5383 public


Apart from the mobile chargers, only the five stations listed above have no province. Counts per province, public and private:

In [8]:
station_counts = (stations.dropna(subset=["province_code"])
                  .pivot_table(index="province_code", columns="access", values="station_id", aggfunc="count", fill_value=0)
                  .rename(columns={"public": "public_stations", "private": "private_stations"})
                  .reset_index())
station_counts.columns.name = None
print("Provinces with stations:", len(station_counts))
station_counts.sort_values("public_stations", ascending=False).head()

Provinces with stations: 81


,province_code,private_stations,public_stations
0,TR100,1692,2742
22,TR510,828,1287
25,TR611,134,782
14,TR411,134,565
6,TR310,188,565


## 5. Population, density, urban share and household size (ADNKS, 2025)

- `population_2025`: resident population at the end of 2025,
- `density_2025`: people per km²,
- `dense_urban_share_2025`: share of people living in dense urban areas (TÜİK urban-rural classification). Six provinces, e.g. Sinop and Tunceli, have no dense urban area at all, so their share is 0,
- `household_size_2025`: average number of people per household.

In [9]:
def adnks_2025(file_name, name, **filters):
    df = read_tuik_csv(file_name)
    df = df[df["Zaman"] == 2025]
    for col, value in filters.items():
        df = df[df[col] == value]
    df = add_province_code(df, "İkamet edilen yer")
    df[name] = to_number(df["Gözlem"])
    return df


pop = adnks_2025("tuik_DF_ADNKS_T35.csv", "population_2025", **{"Yaş Grubu": "Toplam"})
density = adnks_2025("tuik_DF_ADNKS_T20.csv", "density_2025")
hh_size = adnks_2025("tuik_DF_ADNKS_T25.csv", "household_size_2025")

urban = adnks_2025("tuik_DF_ADNKS_T37.csv", "people", Cinsiyet="Toplam")
urban = urban.pivot_table(index="province_code", columns="ADNKS Gösterge", values="people", fill_value=0)
urban["dense_urban_share_2025"] = urban["Yoğun kent nüfus"] / urban.sum(axis=1)

turkiye_population = pop.loc[pop["İkamet edilen yer"] == "Toplam", "population_2025"].iloc[0]
population = (pop.dropna(subset=["province_code"])[["province_code", "population_2025"]]
              .merge(density[["province_code", "density_2025"]], on="province_code")
              .merge(urban["dense_urban_share_2025"].reset_index(), on="province_code")
              .merge(hh_size[["province_code", "household_size_2025"]], on="province_code"))
print(population.shape)
population.head()

(81, 5)


,province_code,population_2025,density_2025,dense_urban_share_2025,household_size_2025
0,TR100,15754053,2943.427010,0.960467,3.091227
1,TR212,422438,69.168928,0.573279,2.604745
2,TR213,379595,59.297845,0.549462,2.648269
3,TR211,1208441,194.302421,0.680891,2.993209
4,TR221,1284517,90.838802,0.420921,2.578938


## 6. GDP per capita (2024) and Human Development Index (2022)

- `gdp_per_capita_try_2024` and `gdp_per_capita_usd_2024`: GDP per person, Turkish lira and US dollars. Only one year is used, so inflation does not affect comparisons between provinces.
- `hdi_2022`: TÜİK Human Development Index (0–1, higher = more developed). It replaces the deprivation index (IMD) of the UK project.

In [10]:
def first(pair_col):
    # (code, label) -> code
    return pair_col.str[0]


gdp_2024 = gdp[(first(gdp["DUZEY_SEVIYE"]) == "3") & (first(gdp["TIME_PERIOD"]) == "2024")].copy()
gdp_2024["province_code"] = first(gdp_2024["REF_AREA"])
gdp_2024["unit"] = first(gdp_2024["UNIT_MEASURE"])
income = (gdp_2024.pivot_table(index="province_code", columns="unit", values="value")
          .rename(columns={"TRY": "gdp_per_capita_try_2024", "USD": "gdp_per_capita_usd_2024"})
          .reset_index())
income.columns.name = None

hdi = read_jsonstat("tuik_DF_BESERI_KALKINMA_ENDEKS.json")
hdi = hdi[(first(hdi["DUZEY_SEVIYE"]) == "3") & (first(hdi["INDICATOR"]) == "E_BKE")
          & (first(hdi["ENDEKS_DEGISIM"]) == "1") & (first(hdi["TIME_PERIOD"]) == "2022")]
income = income.merge(pd.DataFrame({"province_code": first(hdi["REF_AREA"]), "hdi_2022": hdi["value"]}),
                      on="province_code")
print(income.shape)
income.sort_values("hdi_2022").head(3)

(81, 4)


,province_code,gdp_per_capita_try_2024,gdp_per_capita_usd_2024,hdi_2022
60,TRA21,194659.553252,5929.888801,0.747905
75,TRC21,188144.382019,5731.417982,0.751461
68,TRB21,203048.962132,6185.454279,0.764447


## 7. Car parks and building height (BKNA 2021)

From the two Excel tables (data rows 5–85, one per province):
- `households_2021`: households living in a dwelling,
- `no_parking_share_2021`: share of households whose building has **no car park**. This is the main sign that a household cannot charge at home and depends on public chargers,
- `avg_floors_2021`: average number of floors of the building households live in (taller buildings = more flats).

The share is recalculated from the counts rather than taken from the percentage column.

In [11]:
parking = pd.read_excel(TUIK + "tuik_bkna2021_il_otopark.xls", header=None)
floors = pd.read_excel(TUIK + "tuik_bkna2021_il_kat_sayisi.xls", header=None)
turkiye_households = pd.to_numeric(parking.iloc[4, 1])

housing = pd.DataFrame({
    "province_name": parking.iloc[5:86, 0],
    "households_2021": pd.to_numeric(parking.iloc[5:86, 1]),
    "no_parking_2021": pd.to_numeric(parking.iloc[5:86, 3]),
})
housing["no_parking_share_2021"] = housing["no_parking_2021"] / housing["households_2021"]
housing = add_province_code(housing, "province_name")

floors = add_province_code(pd.DataFrame({
    "province_name": floors.iloc[5:86, 0],
    "avg_floors_2021": pd.to_numeric(floors.iloc[5:86, 2]),
}), "province_name")

housing = housing[["province_code", "households_2021", "no_parking_share_2021"]].merge(
    floors[["province_code", "avg_floors_2021"]], on="province_code")
print(housing.shape)
housing.sort_values("no_parking_share_2021").head(3)

(81, 4)


,province_code,households_2021,no_parking_share_2021,avg_floors_2021
5,TR510,1874093,0.204037,7.109446
13,TR424,104358,0.353523,3.803764
37,TR721,424008,0.354923,8.193697


## 8. National monthly series

One row per month for Türkiye, joining three sources:
- **TÜİK, cars by fuel type** (YAKIT_CINSI_V4): `cars_total`, `cars_petrol`, `cars_diesel`, `cars_lpg`, `cars_hybrid`, `cars_electric`, `cars_unknown_fuel`. Electric cars are reported separately from January 2011 and hybrids from January 2012; earlier months are empty.
- **TÜİK, first registrations** (section 3): `new_cars`.
- **EPDK monthly report, September 2026** (typed in from the PDF):
  - `ev_sales_epdk`: electric vehicles sold per month, June 2023 – September 2026 (page 4),
  - `ev_stock_epdk`: electric vehicles registered, all types, September 2025 – September 2026 (page 3),
  - `sockets_ac`, `sockets_dc`, `sockets_total`: commercial charging sockets, September 2025 – September 2026 (page 5).

EPDK sales and stock include a small number of non-car electric vehicles (see notebook 01), so EV shares of the car fleet use `cars_electric` from TÜİK.

In [12]:
fuel = read_tuik_csv("tuik_DF_MOTORLU_KARA_TASIT_YAKIT_CINSI_V4.csv")
fuel = fuel[fuel["Ölçü Birimi"] == "Sayı"].copy()
fuel["value"] = to_number(fuel["Gözlem Değeri"])
fuel = (fuel.pivot_table(index="Zaman", columns="Yakıt Türü", values="value")
        .rename(columns={"Toplam": "cars_total", "Benzinli": "cars_petrol", "Dizel": "cars_diesel", "LPG": "cars_lpg",
                         "Hibrit": "cars_hybrid", "Elektrik": "cars_electric", "Bilinmeyen": "cars_unknown_fuel"}))
fuel.index.name = "month"
fuel.columns.name = None
fuel = fuel[["cars_total", "cars_petrol", "cars_diesel", "cars_lpg", "cars_hybrid", "cars_electric", "cars_unknown_fuel"]]

# EPDK report, September 2026 (typed in from the PDF)
ev_sales_epdk = pd.Series(
    [2999, 5355, 8658, 9889, 8412, 10449, 10181,                                       # Jun-Dec 2023
     8026, 5136, 6843, 5921, 8700, 8551, 8653, 5844, 8548, 8363, 12214, 18010,          # 2024
     12496, 9997, 12274, 11230, 15884, 20729, 23752, 18908, 11053, 13049, 17088, 21898,  # 2025
     15415, 11082, 14654, 16600, 13458, 10863, 15497, 12613, 15319],                     # Jan-Sep 2026
    index=pd.period_range("2023-06", "2026-09", freq="M").astype(str), name="ev_sales_epdk")

epdk_months = pd.period_range("2025-09", "2026-09", freq="M").astype(str)
epdk = pd.DataFrame({
    "ev_stock_epdk": [321710, 334768, 351836, 373733, 389134, 399043, 411796,
                      427486, 440327, 450038, 464873, 476934, 491492],
    "sockets_ac": [19992, 20677, 21422, 22095, 22635, 23142, 23900, 24468, 25125, 25434, 26204, 27044, 27664],
    "sockets_dc": [15010, 15603, 16051, 16713, 17059, 17433, 18038, 18541, 19050, 19663, 20461, 20864, 21307],
    "sockets_total": [35002, 36280, 37473, 38808, 39694, 40575, 41938, 43009, 44175, 45097, 46665, 47908, 48971],
}, index=epdk_months)

national = (fuel.join(new_cars_tr.set_index("month"), how="outer")
            .join(ev_sales_epdk, how="outer")
            .join(epdk, how="outer")
            .rename_axis("month")
            .reset_index())
print(national.shape, national["month"].min(), "to", national["month"].max())
national.tail(3).T

(261, 14) 2005-01 to 2026-09


,258,259,260
month,2026-07,2026-08,2026-09
cars_total,17959336.0,18037907.0,18115875.0
cars_petrol,5571745.0,5594293.0,5616803.0
cars_diesel,5760116.0,5776871.0,5793543.0
cars_lpg,5258752.0,5263305.0,5268278.0
cars_hybrid,873150.0,896270.0,915892.0
cars_electric,460552.0,472157.0,486359.0
cars_unknown_fuel,35021.0,35011.0,35000.0
new_cars,78761,66048,64361
ev_sales_epdk,15497.0,12613.0,15319.0


## 9. One table per province, and checks

All province tables are joined on `province_code`, starting from the list of 81 provinces. Then:
1. Does every source cover all 81 provinces, with no duplicates or missing values?
2. Do province totals add up to the published Türkiye totals?
3. Do the typed-in EPDK figures add up (AC + DC = total)?

In [13]:
province_table = provinces
for table in [vehicles, new_cars, station_counts, population, income, housing]:
    province_table = province_table.merge(table, on="province_code", how="left")

count_cols = ["cars", "all_vehicles", "cars_2025_09", "new_cars_2025", "new_cars_2026_jan_sep",
              "public_stations", "private_stations", "population_2025", "households_2021"]
province_table[count_cols] = province_table[count_cols].astype(int)

# 1. Coverage, duplicates, missing values
for name, table in [("vehicles", vehicles), ("new cars", new_cars), ("stations", station_counts),
                    ("population", population), ("income", income), ("housing", housing)]:
    print(f"{name:11s} rows: {len(table)} | duplicates: {table['province_code'].duplicated().sum()}")
print("Final table:", province_table.shape, "| missing values:", province_table.isna().sum().sum())

vehicles    rows: 81 | duplicates: 0
new cars    rows: 81 | duplicates: 0
stations    rows: 81 | duplicates: 0
population  rows: 81 | duplicates: 0
income      rows: 81 | duplicates: 0
housing     rows: 81 | duplicates: 0
Final table: (81, 21) | missing values: 0


In [14]:
# 2. Province totals vs Türkiye totals
public_total = (stations["access"] == "public").sum()
checks = [
    ("Cars, Sep 2026", province_table["cars"].sum(), turkiye_cars),
    ("Population, 2025", province_table["population_2025"].sum(), turkiye_population),
    ("Households, 2021", province_table["households_2021"].sum(), turkiye_households),
    ("Public stations", province_table["public_stations"].sum(), public_total),
]
for name, provinces_sum, total in checks:
    print(f"{name:18s} provinces: {provinces_sum:>12,.0f} | Türkiye: {total:>12,.0f} | difference: {total - provinces_sum:,.0f}")

# 3. EPDK figures typed in from the PDF
print("\nAC + DC = total in every month:", (national["sockets_ac"] + national["sockets_dc"] == national["sockets_total"])[national["sockets_total"].notna()].all())
print("TÜİK electric cars, Sep 2026:", f"{national.loc[national['month'] == LATEST_MONTH, 'cars_electric'].iloc[0]:,.0f}")

Cars, Sep 2026     provinces:   18,115,875 | Türkiye:   18,115,875 | difference: 0
Population, 2025   provinces:   86,092,168 | Türkiye:   86,092,168 | difference: 0
Households, 2021   provinces:   25,329,833 | Türkiye:   25,329,833 | difference: 0
Public stations    provinces:       12,736 | Türkiye:       12,760 | difference: 24

AC + DC = total in every month: True
TÜİK electric cars, Sep 2026: 486,359


**Results**
- All six province tables have exactly 81 provinces, no duplicates, and the final table has no missing values.
- Cars, population and households add up **exactly** to the Türkiye totals.
- Public stations: 24 are not in any province. These are the 19 mobile charging vans and the 5 Otojet stations with an unclear address (section 4).
- The typed-in EPDK socket figures are consistent (AC + DC = total every month), and the TÜİK electric car count matches the press release (486,359).

## 10. Save the clean tables

In [15]:
province_table.to_csv(PROCESSED + "province_analysis_table.csv", index=False)
stations.to_csv(PROCESSED + "charging_stations.csv", index=False)
national.to_csv(PROCESSED + "national_monthly.csv", index=False)

for f in sorted(os.listdir(PROCESSED)):
    if f.endswith(".csv"):
        print(f"{f:32s} {os.path.getsize(PROCESSED + f) / 1024:8.0f} KB")

charging_stations.csv                3993 KB
national_monthly.csv                   21 KB
province_analysis_table.csv            18 KB


## Summary

| File | One row per | Main columns |
| --- | --- | --- |
| `province_analysis_table.csv` | province (81) | codes and SR-1 region; `cars`, `all_vehicles`, `cars_2025_09`, `new_cars_2025`, `new_cars_2026_jan_sep`; `public_stations`, `private_stations`; `population_2025`, `density_2025`, `dense_urban_share_2025`, `household_size_2025`; `gdp_per_capita_try_2024`, `gdp_per_capita_usd_2024`, `hdi_2022`; `households_2021`, `no_parking_share_2021`, `avg_floors_2021` |
| `charging_stations.csv` | EPDK station (16,613) | `station_id`, `access` (public / private), `mobile`, `brand`, `network_operator`, `station_operator`, `address`, `province_code` |
| `national_monthly.csv` | month, Türkiye (2005-01 to 2026-09) | cars by fuel type (TÜİK), `new_cars` (TÜİK), `ev_sales_epdk`, `ev_stock_epdk`, `sockets_ac`, `sockets_dc`, `sockets_total` (EPDK) |

**Points to remember in the analysis**
- There is still **no electric car count by province**. Province comparisons use public stations per 10,000 cars and per 100,000 people.
- Stations are not sockets: a station can have several sockets. Sockets and AC/DC are only national (EPDK).
- Years differ between sources (vehicles 2026, population 2025, GDP 2024, HDI 2022, housing 2021). Each column name carries its year.
- Istanbul's car count includes company and rental cars registered at head offices.